# Curating research data at scale with a local LLM

This notebook runs the **same curation steps from the workshop run sheet**, but on the full
**1,520-row** dataset (`plants_sites_AB_large.csv` — Sites A and B already combined), using a
local **Qwen** model in Colab instead of a chatbot.

**The key idea: use the LLM for _judgement_, and batch by _unique values_ so it scales.**
Most messy columns have only a handful of distinct values — `Site`, for example, has 14
different spellings of just two real sites. So instead of asking the model about all 1,520 rows,
we ask it once about each *unique* value and map the answer back across every row. That turns
thousands of calls into a few dozen. We only loop row-by-row where a task is genuinely per-row
(reading the free-text `Notes`, and reading the two date columns).

**New columns this notebook creates** (all kept *alongside* the originals):

| Column(s) | How it's produced |
|---|---|
| `Height_cm_cleaned`, `Num_Leaves_cleaned`, `Num_Flowers_cleaned` | **LLM** — fix obvious numeric errors, flag outliers as `check` |
| `Site_cleaned`, `Location_cleaned`, `Quadrat_cleaned`, `Species_cleaned`, `Flowering_cleaned` | **LLM** — standardise spelling / case / format |
| `Condition_cleaned` | **LLM** — map free-text health to a 5-point scale |
| `Notes_themes`, `Data_Quality_Flag`, `Data_Quality_Notes` | **LLM** — read the free-text notes |
| `Plant_Age_Months` | **Python** — date arithmetic is a rule, not a judgement |
| `Plant_ID` | **Python** — a composite identifier built from the cleaned columns |

Everything runs **deterministically** (`do_sample=False`, `temperature=0`) so a re-run gives the
same answers. The model can still be *wrong* — every `check` flag is a row for a human to review.

In [ ]:
#### Before you start
# 1. File > Save a copy in Drive, then work from your copy.
# 2. Connect (top-right) > Change runtime type > T4 GPU > Save.
#    The model needs the GPU. You'll see the RAM / Disk meters once it connects.

In [ ]:
#### Install the AI libraries (about a minute)
!pip install -U bitsandbytes --quiet
!pip install accelerate --quiet
!pip install transformers --quiet

In [ ]:
#### Load the libraries we need
import json
import re
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#### Load the AI model
# Load Large Language Model (LLM)
model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
print("--- Model Loaded Successfully ---")

## Load the data

The file is `plants_sites_AB_large.csv` (1,520 rows). You need to get it into this Colab session
before running the cell below.

**Easiest way — drag and drop:**

1. Click the **folder icon** (📁) in the far-left sidebar to open the **Files** panel.
2. Drag `plants_sites_AB_large.csv` from your computer (Finder / File Explorer) straight into
   that panel and drop it.
3. Wait for the upload to finish — the file appears in the list with a small progress ring while
   it copies. **Don't disconnect the runtime until it's done.**
4. Run the cell below. It will find the file and load it.

> ⚠️ Files uploaded this way live in the **session only** — if the runtime disconnects or you
> restart it, you'll need to drag the file in again.

**No drag-and-drop? Use the upload button instead:** in the same **Files** panel, click the
**Upload** icon (a page with an up-arrow) at the top and pick `plants_sites_AB_large.csv`.

**Prefer not to touch the Files panel at all?** Just run the cell below — if the file isn't
already there, it will pop up a **Choose Files** button so you can select it from your computer.</cell id="cell-5">
</invoke>


In [ ]:
#### Load the combined Site A + Site B dataset
try:
    df = pd.read_csv("plants_sites_AB_large.csv")
except FileNotFoundError:
    from google.colab import files
    print("Please upload plants_sites_AB_large.csv:")
    files.upload()
    df = pd.read_csv("plants_sites_AB_large.csv")

print("Rows, columns:", df.shape)
df.head()

## One reusable function to ask the LLM

Everything below calls `query_llm(...)`. It wraps the same steps as the workshop notebook —
format the prompt in Qwen's chat style, run the model, return just the new text — and runs
**deterministically** so results are repeatable.

`map_unique_values(...)` is the workhorse that makes this scale: give it the unique values of a
column and an instruction, and it returns a `{original -> cleaned}` dictionary we map back onto
all 1,520 rows.

In [ ]:
#### The core helpers

def query_llm(system_prompt, user_prompt, max_new_tokens=512):
    """Send one system + user prompt to Qwen and return only the generated text.
    do_sample=False / temperature=0 -> the same input always gives the same output."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    chat_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    tokenized = tokenizer(
        chat_input, return_tensors="pt", padding=True, truncation=True,
        max_length=tokenizer.model_max_length,
    )
    input_ids = tokenized["input_ids"].to(model.device)
    attention_mask = tokenized["attention_mask"].to(model.device)

    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,        # no randomness
        temperature=0.0,        # no randomness
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.batch_decode(
        generated[:, input_ids.shape[-1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return text.strip()


def extract_json(text):
    """Pull the first JSON object/array out of the reply, tolerating ```json fences."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip().strip("`").strip()
    starts = [i for i in (text.find("{"), text.find("[")) if i != -1]
    if not starts:
        raise ValueError("No JSON found in reply:\n" + text[:300])
    obj, _ = json.JSONDecoder().raw_decode(text[min(starts):])
    return obj


def map_unique_values(values, system_prompt, instruction,
                      batch_size=40, max_new_tokens=1500):
    """Ask the LLM to map each unique value -> cleaned value (or "check").
    Values are sent in small batches so each reply stays short and reliable.
    Returns a dict {original_value_as_string: cleaned_value}."""
    mapping = {}
    values = list(values)
    for start in range(0, len(values), batch_size):
        batch = [str(v) for v in values[start:start + batch_size]]
        user_prompt = (
            instruction
            + "\n\nReturn ONLY a JSON object mapping each original value (as a string) to its "
              'cleaned value. Use "check" when you are unsure. Include every value listed.\n\n'
            + "Values:\n" + json.dumps(batch, ensure_ascii=False)
        )
        try:
            batch_map = extract_json(query_llm(system_prompt, user_prompt, max_new_tokens))
            for k, v in batch_map.items():
                mapping[str(k)] = v
        except Exception as e:
            print(f"  ! could not parse batch starting at {start}: {e}")
    return mapping


def apply_mapping(series, mapping):
    """Map a column through a {value -> cleaned} dict, leaving unmatched values unchanged."""
    def convert(x):
        key = str(x)
        if key in mapping:
            return mapping[key]
        if key.endswith(".0") and key[:-2] in mapping:   # 117.0 vs "117"
            return mapping[key[:-2]]
        return x
    return series.map(convert)

In [ ]:
#### Quick check that the model answers
print(query_llm(
    "You only output valid JSON.",
    'Map each value to lower case. Return a JSON object. Values: ["SITE_A", "Site_b"]',
    max_new_tokens=80,
))

## Step 1 — Numeric columns: fix errors, flag outliers

`Height_cm`, `Num_Leaves`, and `Num_Flowers` contain data-entry errors (decimal shifts, repeated
digits, impossible negatives). Each column has only a few hundred *distinct* values, so we send
the unique values once and get back a `{value -> cleaned}` map. Anything the model is unsure about
comes back as `"check"`.

In [ ]:
#### Clean the three numeric columns
num_system = ("You are a meticulous research data assistant cleaning ecological field "
              "measurements. You only output valid JSON.")

def fmt_num(v):
    f = float(v)
    return str(int(f)) if f == int(f) else str(f)

numeric_specs = {
    "Height_cm": (
        "These are plant heights in centimetres; realistic values are about 0-300.\n"
        "- Correct a clear decimal/typing shift (e.g. 340 -> 34).\n"
        '- Flag any value above 300, below 0, or that you cannot confidently correct as "check".\n'
        "- Return plausible values unchanged."
    ),
    "Num_Leaves": (
        "These are leaf counts; realistic values are about 0-500.\n"
        "- Correct an obvious repeated-digit typo (e.g. 877 -> 87).\n"
        '- Flag any value above 500, below 0, or that you cannot confidently correct as "check".\n'
        "- Return plausible values unchanged."
    ),
    "Num_Flowers": (
        "These are flower counts; they cannot be negative.\n"
        "- Correct a clear sign error (e.g. -3 -> 3).\n"
        '- Flag anything you cannot confidently correct as "check".\n'
        "- Return plausible values (including 0) unchanged."
    ),
}

for col, instruction in numeric_specs.items():
    uniques = [fmt_num(v) for v in sorted(df[col].dropna().unique())]
    print(f"{col}: {len(uniques)} unique values -> asking the model...")
    mapping = map_unique_values(uniques, num_system, f"Column '{col}'.\n{instruction}")
    df[col + "_cleaned"] = apply_mapping(df[col], mapping)
    flagged = (df[col + "_cleaned"].astype(str) == "check").sum()
    print(f"  done. {flagged} rows flagged 'check'.")

df[["Height_cm", "Height_cm_cleaned",
    "Num_Leaves", "Num_Leaves_cleaned",
    "Num_Flowers", "Num_Flowers_cleaned"]].head(10)

## Step 2 — Categorical columns: standardise spelling, case and format

`Site`, `Location`, `Quadrat`, `Species` and `Flowering` are the same handful of real categories
written many different ways. We standardise each to lower-case, underscore-separated values and
flag anything impossible (like a quadrat `q11`) as `check`.

In [ ]:
#### Standardise the categorical columns
cat_system = ("You standardise messy categorical data fields for a researcher. "
              "You only output valid JSON.")

categorical_specs = {
    "Site": "Two real sites exist. Map every value to 'site_a' or 'site_b'.",
    "Location": ("Map every value to 'loc_1', 'loc_2', 'loc_3' or 'loc_4'. "
                 "Flag clearly impossible values as 'check'."),
    "Quadrat": ("Map every value to 'q1' through 'q6' (lower-case, no spaces). "
                "Flag impossible values such as 'q11' as 'check'."),
    "Species": ("These are New Zealand plant species names containing typos. Correct each to its "
                "proper binomial name, then format as lower-case with an underscore between the "
                "two words, e.g. 'leptospermum_scoparium'. Flag unrecognisable names as 'check'."),
    "Flowering": ("This is a yes/no flag. Map to 'yes' or 'no' "
                  "('true'/'1' -> 'yes', 'false'/'0' -> 'no'). Flag anything ambiguous as 'check'."),
}

for col, instruction in categorical_specs.items():
    uniques = [str(v) for v in df[col].dropna().unique()]
    print(f"{col}: {len(uniques)} unique values -> asking the model...")
    mapping = map_unique_values(uniques, cat_system, f"Column '{col}'. {instruction}")
    df[col + "_cleaned"] = apply_mapping(df[col], mapping)
    print(f"  result categories: {sorted(set(df[col + '_cleaned'].dropna().astype(str)))}")

## Step 3 — `Condition`: free text to a 5-point scale

`Condition` is free text ("great condition", "in decline", "could be dead"...). We map it to a
consistent five-point scale from `poor` to `excellent`.

In [ ]:
#### Map Condition to a 5-point scale
cond_instruction = (
    "Column 'Condition' holds free-text descriptions of plant health. Map each description to a "
    "five-point scale, using exactly one of: 'poor', 'fair', 'good', 'very_good', 'excellent' "
    "(poor = worst, excellent = best). Flag anything unclear as 'check'."
)
uniques = [str(v) for v in df["Condition"].dropna().unique() if str(v).strip()]
print(f"Condition: {len(uniques)} unique values -> asking the model...")
cond_map = map_unique_values(uniques, cat_system, cond_instruction)
df["Condition_cleaned"] = apply_mapping(df["Condition"], cond_map)
df[["Condition", "Condition_cleaned"]].drop_duplicates().head(20)

## Step 4 — `Plant_Age_Months`: calculate age from the two date columns

`Date_Planted` and `Date_Measured` are written in mixed formats — `DD/MM/YYYY`, `D/M/YY`,
`DD.MM.YY`, some US `M/D/YY`, and some `DD-Mon` entries that have **no year at all**.

**This step does _not_ use the LLM.** Date arithmetic is a deterministic rule, not a judgement
call, so plain Python is the right tool: it is exact, instant, and reproducible — where a language
model would be slow (~1,500 calls) and could silently misread a date. This is the other half of
the lesson: *use the LLM for judgement, use code for rules.*

The parser uses the day/month values to resolve the format where it can (if a number is > 12 it
must be the day), defaults to day-first for NZ data, and flags `check` when a date can't be read
with confidence (no year, unparseable, or a measurement before planting).

In [ ]:
#### Calculate plant age in months with plain Python (no LLM needed)
import re
from datetime import date

def parse_date(s):
    """Read one messy date string -> a date, or None if it can't be read confidently."""
    s = "" if pd.isna(s) else str(s).strip()
    if not s:
        return None
    if re.search("[A-Za-z]", s):      # e.g. '14-Oct' has no year -> cannot place in time
        return None
    parts = re.split(r"[/.\-]", s)
    if len(parts) != 3:
        return None
    try:
        a, b, c = (int(p) for p in parts)
    except ValueError:
        return None
    year = c + 2000 if c < 100 else c          # '21' -> 2021
    if a > 12 and b <= 12:        day, month = a, b      # >12 must be the day
    elif b > 12 and a <= 12:      month, day = a, b      # US M/D order
    elif a <= 12 and b <= 12:     day, month = a, b      # ambiguous -> day-first (NZ default)
    else:                         return None
    if not (1 <= month <= 12 and 1 <= day <= 31):
        return None
    try:
        return date(year, month, day)
    except ValueError:
        return None

def age_in_months(planted, measured):
    dp, dm = parse_date(planted), parse_date(measured)
    if dp is None or dm is None:
        return "check"
    months = (dm.year - dp.year) * 12 + (dm.month - dp.month)
    if dm.day < dp.day:
        months -= 1
    return months if months >= 0 else "check"      # measured before planted -> flag

df["Plant_Age_Months"] = [age_in_months(p, m)
                          for p, m in zip(df["Date_Planted"], df["Date_Measured"])]

flagged = (df["Plant_Age_Months"].astype(str) == "check").sum()
print(f"Calculated {len(df) - flagged} ages; {flagged} rows flagged 'check'.")
df[["Date_Planted", "Date_Measured", "Plant_Age_Months"]].head(10)

## Step 5 — `Plant_ID`: a composite key

A `Plant_ID` is a **rule**, not a judgement — so we build it directly from the cleaned columns
rather than spending LLM calls on it. Format example: `Sa_L1_Q1_LS`
(Site a, Location 1, Quadrat 1, *Leptospermum scoparium*). Rows with a `check` in any source
column get `check` here too.

In [ ]:
#### Build Plant_ID from the cleaned columns
def make_plant_id(row):
    site, loc, quad, sp = (row["Site_cleaned"], row["Location_cleaned"],
                           row["Quadrat_cleaned"], row["Species_cleaned"])
    if any(pd.isna(x) or str(x) == "check" for x in (site, loc, quad, sp)):
        return "check"
    s = "S" + str(site).split("_")[-1]                          # site_a -> Sa
    l = "L" + "".join(c for c in str(loc) if c.isdigit())       # loc_1  -> L1
    q = "Q" + "".join(c for c in str(quad) if c.isdigit())      # q1     -> Q1
    initials = "".join(w[0].upper() for w in str(sp).split("_")[:2])  # ...-> LS
    return f"{s}_{l}_{q}_{initials}"

df["Plant_ID"] = df.apply(make_plant_id, axis=1)
df["Plant_ID"].head(10)

## Step 6 — Read the free-text `Notes`

The `Notes` field is genuinely qualitative, so we loop. But there are only **66 unique notes**
across all 1,520 rows, so we analyse each note once, cache the answer, and map it back. In one
call per note we get all three outputs:

- `Notes_themes` — one of three botanist's themes
- `Data_Quality_Flag` — `Y`/`N`: does the note suggest the measurements may be unreliable?
- `Data_Quality_Notes` — a short reason when the flag is `Y`

In [ ]:
#### Analyse each unique note once, then map back to all rows
notes_system = ("You are a botanist's research assistant reading short plant field notes. "
                "You only output valid JSON.")

unique_notes = [n for n in df["Notes"].dropna().unique()]
print(f"Analysing {len(unique_notes)} unique notes...")

notes_cache = {}
for i, note in enumerate(unique_notes, 1):
    user = (f'Field note: "{note}"\n\n'
            "Return JSON with exactly these keys:\n"
            '- "theme": ONE of "growth_vigour", "disturbance_pressure", or "measurement_reliability"\n'
            '- "quality_flag": "Y" if the note suggests this row\'s measurements may be unreliable '
            'or need special handling, otherwise "N"\n'
            '- "quality_note": a short reason when the flag is "Y", otherwise an empty string')
    try:
        out = extract_json(query_llm(notes_system, user, max_new_tokens=160))
    except Exception:
        out = {"theme": "check", "quality_flag": "check", "quality_note": ""}
    notes_cache[note] = out
    if i % 10 == 0:
        print(f"  analysed {i} / {len(unique_notes)} notes")

df["Notes_themes"]       = df["Notes"].map(lambda n: notes_cache.get(n, {}).get("theme", ""))
df["Data_Quality_Flag"]  = df["Notes"].map(lambda n: notes_cache.get(n, {}).get("quality_flag", ""))
df["Data_Quality_Notes"] = df["Notes"].map(lambda n: notes_cache.get(n, {}).get("quality_note", ""))
df[["Notes", "Notes_themes", "Data_Quality_Flag", "Data_Quality_Notes"]].head(10)

## Review and save

A quick count of how many rows the model flagged `check` (your human-review list), then we save
the full table — originals plus every new column — to `plants_combined_cleaned.csv`.

In [ ]:
#### Summarise the 'check' flags and save the result
new_cols = ["Height_cm_cleaned", "Num_Leaves_cleaned", "Num_Flowers_cleaned",
            "Site_cleaned", "Location_cleaned", "Quadrat_cleaned", "Species_cleaned",
            "Flowering_cleaned", "Condition_cleaned", "Plant_Age_Months", "Plant_ID",
            "Notes_themes", "Data_Quality_Flag", "Data_Quality_Notes"]

print("Rows flagged 'check' for human review:")
for c in new_cols:
    n = (df[c].astype(str) == "check").sum()
    if n:
        print(f"  {c}: {n}")

df.to_csv("plants_combined_cleaned.csv", index=False)
print("\nSaved plants_combined_cleaned.csv", df.shape)

try:
    from google.colab import files
    files.download("plants_combined_cleaned.csv")
except Exception:
    pass

df[["Species", "Species_cleaned", "Height_cm", "Height_cm_cleaned",
    "Condition", "Condition_cleaned", "Plant_ID", "Notes_themes"]].head(10)

## What to take away

- **Batching by unique values is what makes this affordable.** The categorical, numeric and
  condition columns were a few dozen LLM calls, not 1,520 — and the notes were 66, not 1,520.
- **The LLM is for judgement; code is for rules.** We used the model where values were messy or
  ambiguous, and plain Python for the things that are deterministic — `Plant_Age_Months` (date
  arithmetic) and `Plant_ID` (a composite key). That's faster, exact, and reproducible.
- **`temperature=0` makes the LLM runs repeatable, not correct.** Every `check` flag — and a
  sample of the rest — still needs a human eye.